# The El Farol Bar problem
NOTE: this is a more advanced problem. This example utilizes the power of classes to solve a problem with simulating a game theoretic example, and integrates multiple classes and some plotting. If you are very very new to classes, this might be a challenging notebook.

The El Farol bar problem is a problem in game theory. Every Thursday night, a fixed population want to go have fun at the El Farol Bar, unless it's too crowded.

- If less than 60% of the population go to the bar, they'll all have more fun than if they stayed home.
- If more than 60% of the population go to the bar, they'll all have less fun
  than if they stayed home.

Everyone must decide at the same time whether to go or not, with no knowledge of
others' choices.


It was defined by economist William Brian Arthur in 1994. He wrote a paper about
it, you can read it
[here](https://sites.santafe.edu/~wbarthur/Papers/El_Farol.pdf) if you are
interested

In the paper, every person (called agent) has access to some ways to predict how
many people will come next week (called hypothesis).

This notebook recreates this problem as a way to learn the student about classes
and functions.

# Hypotheses

First, let's load the hypothesis class. Study the file `hypotheses.py` inside
the `elfarol` folder to see what is going on.

NOTE: study means: look at every function and read the all the comments. You can also read the next few paragraphs to get more details about how this is used. Dont proceed untill you get an idea of what is going on here. This is selfstudy! If you dont understand what is going on, ask questions to your teacher!

In [ ]:
import elfarol.hypotheses as hypt

hypotheses = hypt.Hypotheses(n=[1, 3, 6], fixed=True)
hypotheses

A Hypotheses is a class that has 8 different hypotheses implemented as a
function. The types of hypotheses are taken from the paper. When an instance of Hypotheses is created, you can select how many
hypotheses are actually available. 

There are two ways to do this:
- **fixed**: if `fixed` is `True`, you can pass a `List[int]` to `n`. In the
  example above, the hypotheses number 1, 3 and 6 are selected
- **unfixed** else, a random selection of `n` hypotheses is made. So `n=3` will
  select three hypothesis at random. With `n=0` the default naive hypothesis is
  selected, which is 'last_week': the prediction is that there will be as much
  people as last week.

In [ ]:
hypotheses = hypt.Hypotheses(n=0)
hypotheses

In [ ]:
hypotheses = hypt.Hypotheses(n=5)
hypotheses

If we feed a `Hypotheses` a history with a list of integers, every hypotheses
will make a prediction.

In [ ]:
hist = [70, 60, 52, 80, 79]

In [ ]:
for model in hypotheses.models:
    yhat = model(hist[:-1])
    diff = abs(yhat - hist[-1])
    print(f"{model.__name__} : {yhat} ({diff})")

Play around with the `Hypotheses` class until you understand how it works!

# Agents

Next, study the `agents.py` file. First, we have a `BaseAgent` class.

In [ ]:
from elfarol.agents import BaseAgent

baseagent = BaseAgent(threshold=60)
baseagent

The `BaseAgent` implements the methods:
- `_get_threshold`: returns the threshold. 
- `_predict`: retrieves a model and returns a prediction
- `_getmodel`: receives a history, and returns the best model (a Callable that
  inputs a List[int] and outputs a float) that can make a
  prediction for given history. The `BaseAgent` only has the naive model
  available (the `_lastweek` model), so the history is ignored.
- `decide`: an Agent should be able to decide if he goes to the bar, depending
  on the prediction he makes and the threshold he has.

For every agents, the `len` of the agents is the amount of models. For the
`BaseAgent`, this length is 1.

In [ ]:
len(baseagent.hypotheses)

And it has just one model

In [ ]:
baseagent._getmodel(hist=hist[:-1]).__name__

That makes predictions

In [ ]:
baseagent._predict(hist=hist[:-1])

And decisions.

In [ ]:
baseagent.decide(hist)

# Inheritance
Now, why use a `BaseAgent`? It is the most simple and generic case of an Agent.
We can now implement a new type of `BaseAgent`, lets call it the `Agent` class.
That class will inherit all the methods from the `BaseAgent`, but it will
implement a new method for `_getmodel`. 

The `Agent` actually tests all models available to see how well the model would
have performed if he would have used this model last week.

Let's create an `Agent`:

In [ ]:
from elfarol.agents import Agent

agent = Agent(n=10, threshold=60)
agent

We can see it has 8 different hypotheses.

In [ ]:
len(agent.hypotheses)

And, based on the current history, it will select one model:

In [ ]:
agent._getmodel(hist).__name__

And will make a prediction with the selected best model:

In [ ]:
agent._predict(hist)

And can decide, based on that decision

In [ ]:
agent.decide(hist)

# Saturday night

In [ ]:
from scipy import stats
import seaborn as sns

So, now we have everything in place. Let's create a 100 `BaseAgents`

In [ ]:
visitors = 100
agents = [BaseAgent(threshold=60) for _ in range(visitors)]
agents[0]

And simulate 52 saturday nights. We will start with a value of 100, because
everybody was invited to the first evening.

Have a look at the `bar.py` file, the `saturdaynight` method is just a forloop.

In [ ]:
from elfarol.bar import saturdaynight

hist = [100]
for _ in range(52):
    hist = saturdaynight(agents, hist)
sns.lineplot(x=range(len(hist)), y=hist)

Interesting! What is going on: last week, there where a 100 visitors. They all
have just one model, that predicts: next week there will be the same amount of
people. 100 is too much, so nobody comes. The week after that, they all predict
no one will come (based on last week) so everybody comes. This will bounce
between 0 and 100.


Well, maybe the problem is the threshold. People will have different preferences, right?


In [ ]:
tresholds = stats.poisson(60).rvs(visitors)
tresholds

In [ ]:
agents = [BaseAgent(threshold=t) for t in tresholds]
hist = [100]
for _ in range(52):
    hist = saturdaynight(agents, hist)
sns.lineplot(x=range(len(hist)), y=hist)

Same result....

Now, let's switch to the `Agent` that can pick between models.
We will give the agent 2 different models, which every agent will pick at random, and every week he will evaluate the performance of both models and stick to the model that performed best.

In [ ]:
agents = [Agent(n=2) for _ in range(visitors)]
hist = [100]
for _ in range(52):
    hist = saturdaynight(agents, hist)
sns.lineplot(x=range(len(hist)), y=hist)

Ok, this is starting to look a bit more natural. At a minumum we see chaotic behavior emerge from our simulation. Let's make the amount of
hypotheses vary between every `Agent`

In [ ]:
N = stats.poisson(2).rvs(visitors)
sns.histplot(N)

In [ ]:
agents = [Agent(n=int(n)) for n in N]
hist = [100]
for _ in range(52 * 3):
    hist = saturdaynight(agents, hist)
sns.lineplot(x=range(len(hist)), y=hist)

And even more variation:

In [ ]:
N = stats.poisson(3).rvs(visitors)
agents = [Agent(n=int(n)) for n in N]
hist = [100]
for _ in range(52 * 3):
    hist = saturdaynight(agents, hist)
sns.lineplot(x=range(len(hist)), y=hist)

# Excercise 1
Add another type of `Agent`, the `MoodyAgent`. This new type of agent will have
a varying threshold. So:

- implement a new type that inherits from `BaseAgent` called `MoodyAgent`
- create noise with `np.random.randint` with parameters `low=-9` and `high=10` (this
  will give a mean of 0), everytime
  `._get_threshold` is called and add this noise to the threshold
- modify the `.decide` function for the `MoodyAgent`, such that the in the
  comparison between the predicted `yhat` and the threshold, the function
  `._get_threshold` is called and every week the threshold has a slight
  variation (based on the mood of the agent)

NOTE: the answers are in the solution.py file

In [ ]:
# %load_ext autoreload
# %autoreload 2
# uncomment these two lines with autoreload if you want to develop inside your notebook.
# having autoreload enabled will let changes in your class show up if you reload the class.
# If this is not enabled, your class will stay the same after loading it from memory.
from elfarol.solution import MoodyAgent
import seaborn as sns

In [ ]:
import numpy as np

np.random.seed(123)
agent = MoodyAgent(2, threshold=60)
agent

In [ ]:
thresholds = [agent._get_threshold(agent.threshold) for _ in range(int(1e3))]

In [ ]:
sns.histplot(thresholds)

In [ ]:
assert np.abs(np.mean(thresholds) - agent.threshold) < 0.5
assert np.std(thresholds) > 5

# Excercise 2

Inside the `bar.py` file, create two classes:

- A `BaseExperiment` class that implements the `saturdaynight` and `simulate`
  functions, but without plotting
- A property of the object, that stores the history in self.hist. Update this
  self.history with new experiments.
- Add a method `.reset` that resets the history
- a property `self.agents` that stores the agents list at creation
- `.simulate` wont need `agents` as an argument, but can use the `self.agents`

With this Base class, add:
- An `Experiment` class that inherits from this `BaseExperiment` and implements
  multiple visualisation options

Add as visualisations:
- the simple lineplot  of the last experiments
- a plot function that creates a histogram for the amount of hypotheses of the
  agents
- a barplot that will collect the distribution of the hypotheses used. This
  should answer the question: which hypotheses are used most often by the
  agents, during the length of the experiment?

In [ ]:
from elfarol.solution import Experiment
from elfarol.solution import MoodyAgent
from scipy import stats

In [ ]:
agent = MoodyAgent(8)
agent

But after the agent made a decision

In [ ]:
agent.decide([60, 65, 55])
agent.log

The log keeps track of which model has been used.

In [ ]:
visitors = 100
N = stats.poisson(2).rvs(visitors)

In [ ]:
agents = [MoodyAgent(int(n)) for n in N]
agents[0]

# Experiment Class

In [ ]:
exp = Experiment(agents=agents)

We can run experiments for 52 weeks

In [ ]:
hist = exp.simulate(k=52)
print(hist)

And call a line plot

In [ ]:
exp.line()

Visualize the amount of hypotheses for agents

In [ ]:
exp.hist_n_hypotheses()

Check the distribution of the models available

In [ ]:
exp.bar_hypt_population(figsize=(6, 8))

To the MoodyAgent, I added a .log property. Everytime a decision is made, the
name of the model used for the decision is added to the log.

In [ ]:
len(exp.agents[0].log)

After 52 weeks, we see that the first agents has used 52 times a model, which
makes sense. We can use a list comprehension to flatten this list

In [ ]:
models_used = [name for x in exp.agents for name in x.log]
len(models_used)

And we get 52*100 models used.

In [ ]:
exp.bar_log()